# Student Activity: Calculate Gini for All Splits

## Dataset: Campus Outdoor Event Decision

**Problem:** A college event committee decides whether an outdoor event should **Proceed** or be **Postponed** based on the weather, venue, registration level, and equipment status.

This is a small, tennis-style categorical classification dataset, but it uses a different and original scenario.

### Assignment requirements
For every candidate split, calculate:
1. Parent Gini impurity
2. Left child Gini impurity
3. Right child Gini impurity
4. Weighted child Gini impurity
5. Gini reduction

**Split convention used:** For each categorical feature, each distinct category is tested as a binary split: `feature == category` (left child) versus `feature != category` (right child). This makes every candidate split explicit and comparable.

## 1. Import the required library

We use pandas to store and process the dataset. The Gini calculations are implemented directly so that each step is visible.

In [ ]:
import pandas as pd

## 2. Create the dataset

In [ ]:
data = [
    ["Sunny", "Open_Ground", "High", "Available", "Proceed"],
    ["Sunny", "Open_Ground", "Low", "Available", "Proceed"],
    ["Rainy", "Open_Ground", "High", "Available", "Postpone"],
    ["Cloudy", "Auditorium", "High", "Available", "Proceed"],
    ["Rainy", "Open_Ground", "Low", "Unavailable", "Postpone"],
    ["Cloudy", "Open_Ground", "High", "Available", "Proceed"],
    ["Windy", "Open_Ground", "High", "Available", "Postpone"],
    ["Sunny", "Auditorium", "Low", "Available", "Proceed"],
    ["Rainy", "Auditorium", "High", "Available", "Proceed"],
    ["Windy", "Open_Ground", "Low", "Unavailable", "Postpone"],
    ["Cloudy", "Auditorium", "Low", "Available", "Proceed"],
    ["Sunny", "Open_Ground", "High", "Unavailable", "Postpone"],
    ["Rainy", "Open_Ground", "High", "Unavailable", "Postpone"],
    ["Windy", "Auditorium", "Low", "Available", "Proceed"],
]

columns = [
    "Weather",
    "Venue",
    "Registration_Level",
    "Equipment_Status",
    "Event_Decision"
]

df = pd.DataFrame(data, columns=columns)
df

## 3. Understand the dataset

- **Weather:** Sunny, Rainy, Cloudy, Windy
- **Venue:** Open_Ground or Auditorium
- **Registration_Level:** High or Low
- **Equipment_Status:** Available or Unavailable
- **Event_Decision:** Proceed or Postpone (target column)

The target column is `Event_Decision`. The remaining columns are input features.

In [ ]:
print("Dataset shape:", df.shape)
print("\\nClass counts:")
print(df["Event_Decision"].value_counts())
print("\\nUnique values in each feature:")
for column in df.columns[:-1]:
    print(column, ":", sorted(df[column].unique()))

## 4. Define the Gini impurity function

For a node containing classes with proportions \(p_1, p_2, \ldots, p_k\):

\[
Gini = 1 - \sum_{i=1}^{k} p_i^2
\]

A Gini value of 0 means the node contains only one class.

In [ ]:
def gini_impurity(series):
    if len(series) == 0:
        return 0.0

    probabilities = series.value_counts(normalize=True)
    return 1 - (probabilities ** 2).sum()

parent_gini = gini_impurity(df["Event_Decision"])
print(f"Parent Gini Impurity: {parent_gini:.4f}")

## 5. Calculate the Gini values for one split

For each candidate category:
- **Left child:** rows where the feature equals the category.
- **Right child:** all remaining rows.
- **Weighted child Gini:** \(\frac{N_L}{N}Gini(L) + \frac{N_R}{N}Gini(R)\)
- **Gini reduction:** \(Gini(parent) - Gini(weighted)\)

In [ ]:
def calculate_split_gini(dataframe, feature, category, target):
    left = dataframe[dataframe[feature] == category]
    right = dataframe[dataframe[feature] != category]

    parent = gini_impurity(dataframe[target])
    left_gini = gini_impurity(left[target])
    right_gini = gini_impurity(right[target])

    total = len(dataframe)
    weighted_child_gini = (
        (len(left) / total) * left_gini
        + (len(right) / total) * right_gini
    )

    gini_reduction = parent - weighted_child_gini

    return {
        "Feature": feature,
        "Split": f"{feature} == {category}",
        "Left_Count": len(left),
        "Right_Count": len(right),
        "Parent_Gini": parent,
        "Left_Gini": left_gini,
        "Right_Gini": right_gini,
        "Weighted_Child_Gini": weighted_child_gini,
        "Gini_Reduction": gini_reduction
    }

## 6. Calculate every candidate split

The loop tests every distinct category of every input feature. Therefore, no category-based candidate split is skipped.

In [ ]:
target = "Event_Decision"
features = [column for column in df.columns if column != target]

results = []

for feature in features:
    for category in sorted(df[feature].unique()):
        result = calculate_split_gini(df, feature, category, target)
        results.append(result)

results_df = pd.DataFrame(results)
results_df

## 7. Display the five required values clearly

This table contains the exact five Gini values requested in the assignment, along with the split name and child sizes.

In [ ]:
required_columns = [
    "Feature",
    "Split",
    "Parent_Gini",
    "Left_Gini",
    "Right_Gini",
    "Weighted_Child_Gini",
    "Gini_Reduction"
]

display(
    results_df[required_columns]
    .sort_values("Gini_Reduction", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

## 8. Identify the best split

The best split is the candidate with the **highest Gini reduction** (equivalently, the lowest weighted child Gini when the parent is fixed).

In [ ]:
best_split = results_df.loc[results_df["Gini_Reduction"].idxmax()]

print("Best split:", best_split["Split"])
print(f"Parent Gini Impurity:       {best_split['Parent_Gini']:.4f}")
print(f"Left Child Gini Impurity:   {best_split['Left_Gini']:.4f}")
print(f"Right Child Gini Impurity:  {best_split['Right_Gini']:.4f}")
print(f"Weighted Child Gini:        {best_split['Weighted_Child_Gini']:.4f}")
print(f"Gini Reduction:             {best_split['Gini_Reduction']:.4f}")

## 9. Final observation

A good split separates the two event decisions into purer child groups. The best candidate is the one with the largest Gini reduction.

**Assignment conclusion:** The results table above evaluates every category-based binary split in the dataset. Use the displayed values to explain which split is best and why.